# Olist E-Commerce Project Starter

Use this starter notebook for the BAN 6003 final project option: **Olist Brazilian E-Commerce**.

Run cells from top to bottom first. Then replace starter notes with your own project decisions, checks, and interpretation.

Start each milestone by reading the matching Canvas milestone assignment page. The Canvas page tells you exactly what evidence and submission items are required; this notebook gives you starter spaces to build that work.



## Before You Start: Key Project Hints

This dataset is relational. Your main job is to preserve the row meaning of your final ABT.

- A common final ABT is **one row per order**, identified by `order_id`.
- Aggregate `order_items`, `order_payments`, and `order_reviews` to `order_id` before joining them to `orders`.
- Join `products`, category translations, and sellers to item-level data before aggregating those fields to order level.
- `customer_id` identifies the customer record attached to one order; `customer_unique_id` may repeat because one person can place multiple orders.
- Check row counts and duplicate `order_id` values after every major merge.
- Define targets only where the outcome is observable. An undelivered order is not automatically an on-time order, and a missing review is not a low review.
- State when a prediction would be made, then exclude fields that would not be available at that time.


## Setup

In [4]:
from pathlib import Path
import pandas as pd
import numpy as np

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "README.md").exists() and (REPO_ROOT.parent / "README.md").exists():
    REPO_ROOT = REPO_ROOT.parent

DATA_DIR = REPO_ROOT / "data"
OUTPUT_DIR = REPO_ROOT / "outputs"
REPORTS_DIR = REPO_ROOT / "reports"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 100)


## Load Data

Read each provided table. Keep the original row meaning in mind before joining tables.

In [5]:
orders = pd.read_csv(DATA_DIR / 'orders.csv', parse_dates=[
    'order_purchase_timestamp', 'order_approved_at',
    'order_delivered_carrier_date', 'order_delivered_customer_date',
    'order_estimated_delivery_date'
])
customers = pd.read_csv(DATA_DIR / 'customers.csv')
items = pd.read_csv(DATA_DIR / 'order_items.csv', parse_dates=['shipping_limit_date'])
payments = pd.read_csv(DATA_DIR / 'order_payments.csv')
reviews = pd.read_csv(DATA_DIR / 'order_reviews.csv', parse_dates=['review_creation_date', 'review_answer_timestamp'])
products = pd.read_csv(DATA_DIR / 'products.csv')
sellers = pd.read_csv(DATA_DIR / 'sellers.csv')
geo = pd.read_csv(DATA_DIR / 'geolocation_zip_prefix.csv')
translations = pd.read_csv(DATA_DIR / 'product_category_name_translation.csv')

tables = {
    'orders': orders,
    'customers': customers,
    'items': items,
    'payments': payments,
    'reviews': reviews,
    'products': products,
    'sellers': sellers,
    'geolocation_zip_prefix': geo,
    'translations': translations,
}

## Basic Data Profile

Use this section for your first pass through row counts, columns, data types, missingness, duplicates, and suspicious values.

In [6]:
for name, df in tables.items():
    print(f'\n{name}: {df.shape[0]:,} rows x {df.shape[1]:,} columns')
    display(df.head(5))


orders: 25,000 rows x 8 columns


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,3b697a20d9e427646d92567910af6d57,355077684019f7f60a031656bd7262b8,delivered,2016-10-03 09:44:50,2016-10-06 15:50:54,2016-10-23 14:02:13,2016-10-26 14:02:13,2016-10-27
1,cd3b8574c82b42fc8129f6d502690c3e,7812fcebfc5e8065d31e1bb5f0017dae,delivered,2016-10-03 22:31:31,2016-10-04 10:19:23,2016-10-08 10:34:01,2016-10-14 16:08:00,2016-11-23
2,1aecadf4362edaca7fa033e882076c8d,e81a9f176936e3124dfd90c483bf3289,canceled,2016-10-04 10:05:45,2016-10-04 10:26:40,NaT,NaT,2016-11-24
3,3f72d2b757e725cd48a4726f831c7789,e0de521fbb397bd7e83d079fe66357b5,delivered,2016-10-04 13:15:46,2016-10-04 13:47:04,2016-10-10 16:45:50,2016-10-17 11:25:59,2016-11-28
4,63638a6806d67773f3adba8534553fff,16e14c1e6e050fe6730c961ff638ca23,delivered,2016-10-04 13:22:56,2016-10-04 13:47:45,2016-11-17 15:53:01,2016-11-25 13:17:37,2016-11-28



customers: 25,000 rows x 5 columns


,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP
2,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP
3,fd826e7cf63160e536e0908c76c3f441,addec96d2e059c80c30fe6871d30d177,4534,sao paulo,SP
4,5adf08e34b2e993982a47070956c5c65,1175e95fb47ddff9de6b2b06188f7e0d,81560,curitiba,PR



items: 28,343 rows x 7 columns


...

,seller_id,seller_zip_code_prefix,seller_city,seller_state
0,3442f8959a84dea7ee197c632cb2df15,13023,campinas,SP
1,d1b65fc7debc3361ea86b5f14c68d2e2,13844,mogi guacu,SP
2,c0f3eea2e14555b6faeea3dd58c1b1c3,4195,sao paulo,SP
3,e49c26c3edfa46d227d5121a6b6e4d37,55325,brejao,PE
4,1b938a7ec6ac5061a66a3766e0e75f90,16304,penapolis,SP



geolocation_zip_prefix: 10,009 rows x 6 columns


,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state,geolocation_records
0,1001,-23.550190,-46.634024,sao paulo,SP,26
1,1004,-23.549799,-46.634757,sao paulo,SP,22
2,1005,-23.549456,-46.636733,sao paulo,SP,25
3,1006,-23.550102,-46.636137,sao paulo,SP,9
4,1008,-23.546002,-46.635886,sao paulo,SP,16



translations: 71 rows x 2 columns


,product_category_name,product_category_name_english
0,beleza_saude,health_beauty
1,informatica_acessorios,computers_accessories
2,automotivo,auto
3,cama_mesa_banho,bed_bath_table
4,moveis_decoracao,furniture_decor


In [7]:
profile_rows = []
for name, df in tables.items():
    profile_rows.append({
        'table': name,
        'rows': len(df),
        'columns': df.shape[1],
        'duplicate_rows': int(df.duplicated().sum()),
        'missing_cells': int(df.isna().sum().sum()),
    })
pd.DataFrame(profile_rows)

,table,rows,columns,duplicate_rows,missing_cells
0,orders,25000,8,0,1266
1,customers,25000,5,0,0
2,items,28343,7,0,0
3,payments,26154,5,0,0
4,reviews,24934,7,0,36618
5,products,13344,9,0,924
6,sellers,2234,4,0,0
7,geolocation_zip_prefix,10009,6,0,0
8,translations,71,2,0,0


### Key and Granularity Audit

Confirm the expected key for each table before planning joins. A duplicate `order_id` is expected in item, payment, and review detail tables because those tables can contain several records for one order. Their more detailed keys should still be checked. The provided geolocation table is already summarized to one row per zip-code prefix.


In [8]:
key_audit = pd.DataFrame([
    {"table": "orders", "expected_key": "order_id", "duplicate_keys": orders["order_id"].duplicated().sum(), "missing_key_values": orders["order_id"].isna().sum()},
    {"table": "customers", "expected_key": "customer_id", "duplicate_keys": customers["customer_id"].duplicated().sum(), "missing_key_values": customers["customer_id"].isna().sum()},
    {"table": "order_items", "expected_key": "order_id + order_item_id", "duplicate_keys": items.duplicated(["order_id", "order_item_id"]).sum(), "missing_key_values": items[["order_id", "order_item_id"]].isna().any(axis=1).sum()},
    {"table": "order_payments", "expected_key": "order_id + payment_sequential", "duplicate_keys": payments.duplicated(["order_id", "payment_sequential"]).sum(), "missing_key_values": payments[["order_id", "payment_sequential"]].isna().any(axis=1).sum()},
    {"table": "order_reviews", "expected_key": "review record (review_id may repeat)", "duplicate_keys": reviews.duplicated().sum(), "missing_key_values": reviews["order_id"].isna().sum()},
    {"table": "products", "expected_key": "product_id", "duplicate_keys": products["product_id"].duplicated().sum(), "missing_key_values": products["product_id"].isna().sum()},
    {"table": "sellers", "expected_key": "seller_id", "duplicate_keys": sellers["seller_id"].duplicated().sum(), "missing_key_values": sellers["seller_id"].isna().sum()},
    {"table": "geolocation_zip_prefix", "expected_key": "geolocation_zip_code_prefix", "duplicate_keys": geo["geolocation_zip_code_prefix"].duplicated().sum(), "missing_key_values": geo["geolocation_zip_code_prefix"].isna().sum()},
    {"table": "translations", "expected_key": "product_category_name", "duplicate_keys": translations["product_category_name"].duplicated().sum(), "missing_key_values": translations["product_category_name"].isna().sum()},
])

display(key_audit)
print(f"Item rows sharing an order_id: {items['order_id'].duplicated().sum():,}")
print(f"Payment rows sharing an order_id: {payments['order_id'].duplicated().sum():,}")
print(f"Review rows sharing an order_id: {reviews['order_id'].duplicated().sum():,}")
print(f"Repeated customer_unique_id values: {customers['customer_unique_id'].duplicated().sum():,}")


,table,expected_key,duplicate_keys,missing_key_values
0,orders,order_id,0,0
1,customers,customer_id,0,0
2,order_items,order_id + order_item_id,0,0
3,order_payments,order_id + payment_sequential,0,0
4,order_reviews,review record (review_id may repeat),0,0
5,products,product_id,0,0
6,sellers,seller_id,0,0
7,geolocation_zip_prefix,geolocation_zip_code_prefix,0,0
8,translations,product_category_name,0,0


Item rows sharing an order_id: 3,552
Payment rows sharing an order_id: 1,154
Review rows sharing an order_id: 137
Repeated customer_unique_id values: 236


## Milestone 1 Starter: Initial Profile and Cleaning Plan

Complete this by the first project milestone.

Write notes on:

- Business problem and decision context
- Raw tables and row meaning
- Initial row and column counts
- Data type issues
- Missing values and duplicates
- Suspicious values
- Proposed target/outcome
- Early ethics or governance concerns

Important Olist notes:

- Keep a table inventory: what does one row mean in each file?
- `order_items`, `order_payments`, and `order_reviews` are not guaranteed to be one row per order.
- Missing review comments are common and should be described, not automatically deleted.
- Product categories may need translation using `product_category_name_translation.csv`.

### Read the Canvas milestone page first

Before you start this section, read the matching Canvas milestone assignment page. The Canvas page is the authoritative checklist for what you need to submit, how the milestone is graded, and what evidence should appear in your notebook/report. Use this starter section as a workspace, not as a replacement for the Canvas instructions.



### ooo

- Tables/files used: orders, customers, order_items, products, sellers, and order_payments csv files will be used for the analysis. product_category_name_translation.cvs may also be used to translate product categories. orders.csv will serve as the base table because the final analysis will focus on one row per order.

- Early data quality issues: Initial profiling identified several data quality issues that may need further investigation. ZIP code prefix columns in the customers, sellers, and geolocation tables are stored as integers even though they represent geographic codes. Some delivery-related date columns in orders contain missing values. Some item prices, freight values, and payment values may be unusually high or require further inspection. The reviews table also contains 56 repeated review_id values, although these should not automatically be treated as duplicates. Categorical values were checked using value_counts(), and no obvious inconsistencies were identified.

- Initial cleaning plan: ZIP code prefix columns will be converted to string type. Missing delivery-related dates will be compared with order_status before deciding how to handle them. Unusual price, freight, and payment values will be investigated before deciding whether cleaning is necessary. Repeated review_id records will also be inspected with their associated order_id values before any duplicate handling is performed.

- Possible target/outcome: The primary analytical target is late-delivery risk at the order level, while the potential business users are fulfillment, operations, and supply chain teams. I expect longer shipping distances and cross-state deliveries to be associated with a higher risk of late delivery. Product and order characteristics, such as product size, weight, and number of items, may also be related to delivery risk. In addition, some geographic areas may experience higher late-delivery rates than others.

- Ethics, privacy, or governance concerns: This data analysis should be conducted responsibly by avoiding unfair assumptions about customers or sellers. Customer and seller location information should be handled carefully to protect privacy, and all data cleaning and transformation decisions should be clearly documented to maintain proper data governance.

## 1. Business question and Analytical question

- Business or research question: What factors are associated with late delivery in an e-commerce marketplace, and how can these insights help identify higher-risk orders and geographic areas that may benefit from additional fulfillment capacity?

- Why this question matters: Late delivery is an important supply chain and operations issue because it can affect both customer experience and fulfillment efficiency. I became interested in this topic after speaking with H-E-B representatives at a recent career fair, where I learned about the company’s ongoing supply chain expansion, including new fulfillment, warehouse, and distribution facilities. We also discussed how business teams use data when making expansion decisions, such as evaluating where additional capacity may be needed and whether a location can effectively support operations. This made me curious about how I could approach a similar problem as a data analyst.
Through this project, I want to explore which factors are associated with late delivery, which orders may have higher delivery risk, and whether there are geographic patterns in delivery challenges. Although the dataset is not related to H-E-B, it allows me to explore how similar types of data analysis can be used to better understand fulfillment and supply chain operations.



## 2. Understanding the Olist Data

Before starting the data profiling, I first reviewed the Olist dataset documentation to understand what each table represents and how the tables are related.

The dataset contains multiple tables covering different parts of the e-commerce process:

- `orders`: one row represents one order and contains order status and purchase/delivery timestamps.
- `customers`: one row represents an order-specific customer record and contains customer location information.
- `order_items`: one row represents an item within an order and contains product, seller, price, and freight information.
- `order_payments`: one row represents a payment record associated with an order.
- `order_reviews`: one row represents a review record associated with an order.
- `products`: one row represents a product and contains product category, weight, and dimension information.
- `sellers`: one row represents a seller and contains seller location information.
- `geolocation`: contains geographic information based on ZIP-code prefixes.
- `category_translation`: provides English translations for product category names.

These tables have different levels of detail. For example, the `orders` table contains one row per order, while `order_items` and `order_payments` can contain multiple rows for the same order.

Since my project focuses on late delivery, the `orders` table will be the main starting point for the analysis. The other tables may later provide additional information about customers, products, sellers, and geographic locations that could help explain delivery patterns.

Before deciding how these tables should be cleaned or combined, I first need to examine their structure and data quality.

## 3. Initial Data Profiling

In [9]:
# Milestone 1 starter checks
for name, df in tables.items():
    print(f'\n{name}')
    display(df.dtypes.to_frame('dtype').head(20))
    display(df.isna().mean().sort_values(ascending=False).head(10).to_frame('missing_rate'))


orders


,dtype
order_id,str
customer_id,str
order_status,str
order_purchase_timestamp,datetime64[us]
order_approved_at,datetime64[us]
order_delivered_carrier_date,datetime64[us]
order_delivered_customer_date,datetime64[us]
order_estimated_delivery_date,datetime64[us]


,missing_rate
order_delivered_customer_date,0.03100
order_delivered_carrier_date,0.01808
order_approved_at,0.00156
order_id,0.00000
customer_id,0.00000
order_status,0.00000
order_purchase_timestamp,0.00000
order_estimated_delivery_date,0.00000



customers


,dtype
customer_id,str
customer_unique_id,str
customer_zip_code_prefix,int64
customer_city,str
customer_state,str


...

,dtype
geolocation_zip_code_prefix,int64
geolocation_lat,float64
geolocation_lng,float64
geolocation_city,str
geolocation_state,str
geolocation_records,int64


,missing_rate
geolocation_zip_code_prefix,0.0
geolocation_lat,0.0
geolocation_lng,0.0
geolocation_city,0.0
geolocation_state,0.0
geolocation_records,0.0



translations


,dtype
product_category_name,str
product_category_name_english,str


,missing_rate
product_category_name,0.0
product_category_name_english,0.0


### 1) Profiling table [orders] 

### Note
- One row in the Orders table represents one order. 
- The table has 25,000 rows and 8 columns. 
- The data types appear appropriate, but there are some missing values in the date columns. 
- order_id is unique with no missing values, and there are no fully duplicated rows. 
- order_status has 8 unique categories. 
- The min and max values of order_purchase_timestamp show a reasonable date range, with no obvious suspicious values identified so far. 
- Further investigation is needed for the missing values and other possible data quality issues.

In [10]:
orders.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,3b697a20d9e427646d92567910af6d57,355077684019f7f60a031656bd7262b8,delivered,2016-10-03 09:44:50,2016-10-06 15:50:54,2016-10-23 14:02:13,2016-10-26 14:02:13,2016-10-27
1,cd3b8574c82b42fc8129f6d502690c3e,7812fcebfc5e8065d31e1bb5f0017dae,delivered,2016-10-03 22:31:31,2016-10-04 10:19:23,2016-10-08 10:34:01,2016-10-14 16:08:00,2016-11-23
2,1aecadf4362edaca7fa033e882076c8d,e81a9f176936e3124dfd90c483bf3289,canceled,2016-10-04 10:05:45,2016-10-04 10:26:40,NaT,NaT,2016-11-24
3,3f72d2b757e725cd48a4726f831c7789,e0de521fbb397bd7e83d079fe66357b5,delivered,2016-10-04 13:15:46,2016-10-04 13:47:04,2016-10-10 16:45:50,2016-10-17 11:25:59,2016-11-28
4,63638a6806d67773f3adba8534553fff,16e14c1e6e050fe6730c961ff638ca23,delivered,2016-10-04 13:22:56,2016-10-04 13:47:45,2016-11-17 15:53:01,2016-11-25 13:17:37,2016-11-28


In [11]:
orders.tail()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
24995,21a00b08cbeb5716bbb66105e3dbd850,002b5342c72978cf0aba6aae1f5d5293,canceled,2018-09-10 17:51:25,NaT,NaT,NaT,2018-09-20
24996,87ae60ef8b08ae0e5f903cacb53a6904,fea4d57ed3a45455f89c25ef3dae8ee8,canceled,2018-09-11 16:45:54,NaT,NaT,NaT,2018-09-26
24997,ea844c92cf978ea23321fa7fe5871761,6ddc45ad08e6a2c885ca2255fc52d6df,canceled,2018-09-13 09:56:12,NaT,NaT,NaT,2018-09-27
24998,392ed9afd714e3c74767d0c4d3e3f477,2823ffda607a2316375088e0d00005ec,canceled,2018-09-29 09:13:03,NaT,NaT,NaT,2018-10-15
24999,616fa7d4871b87832197b2a137a115d2,bf6181a85bbb4115736c0a8db1a53be3,canceled,2018-10-01 15:30:09,NaT,NaT,NaT,2018-10-23


In [12]:
orders.shape

(25000, 8)

In [13]:
orders.info()

<class 'pandas.DataFrame'>
RangeIndex: 25000 entries, 0 to 24999
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   order_id                       25000 non-null  str           
 1   customer_id                    25000 non-null  str           
 2   order_status                   25000 non-null  str           
 3   order_purchase_timestamp       25000 non-null  datetime64[us]
 4   order_approved_at              24961 non-null  datetime64[us]
 5   order_delivered_carrier_date   24548 non-null  datetime64[us]
 6   order_delivered_customer_date  24225 non-null  datetime64[us]
 7   order_estimated_delivery_date  25000 non-null  datetime64[us]
dtypes: datetime64[us](5), str(3)
memory usage: 3.3 MB


In [14]:
orders.isna().sum()

order_id                           0
customer_id                        0
order_status                       0
order_purchase_timestamp           0
order_approved_at                 39
order_delivered_carrier_date     452
order_delivered_customer_date    775
order_estimated_delivery_date      0
dtype: int64

In [15]:
orders["order_status"].nunique()

8

In [16]:
orders["order_status"].value_counts()

order_status
delivered      24222
shipped          309
unavailable      167
canceled         154
invoiced          75
processing        71
approved           1
created            1
Name: count, dtype: int64

In [17]:
orders.duplicated().sum()

np.int64(0)

In [18]:
orders["order_id"].duplicated().sum()

np.int64(0)

In [19]:
orders["order_purchase_timestamp"].min()

Timestamp('2016-10-03 09:44:50')

In [20]:
orders["order_purchase_timestamp"].max()

Timestamp('2018-10-01 15:30:09')

### 2) Profiling table [customers]
### Note
- One row in the Customers table represents a customer record for one order. If the same customer places another order, a new customer_id is created.
- The table has 25,000 rows and 5 columns. 
- There are no missing values or fully duplicated rows, and customer_id is unique with no duplicates. However, customer_unique_id has repeated values because the same customer can place multiple orders. 
- customer_zip_code_prefix is currently stored as an integer, but it represents a geographic code rather than a numeric measurement, so converting it to a string should be considered during cleaning.

In [21]:
customers.head()

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP
2,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP
3,fd826e7cf63160e536e0908c76c3f441,addec96d2e059c80c30fe6871d30d177,4534,sao paulo,SP
4,5adf08e34b2e993982a47070956c5c65,1175e95fb47ddff9de6b2b06188f7e0d,81560,curitiba,PR


In [22]:
customers.tail()

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
24995,c156d63bdfce1d456bd43cf1c4dadfca,e5cf9da72b751849b7b217f7a943d0fa,9060,santo andre,SP
24996,a1b5ca506b592bb72d4caadcbfe71385,c811311629fd3d76f5ca5eeb2abfa7e8,17511,marilia,SP
24997,935993f47af1ed7d0715c26b686341c5,4452b8ef472646c4cc042cb31a291f3b,12236,sao jose dos campos,SP
24998,592b8900e0e8325027d885e6d30d0283,57c2cfb4a80b13ed19b5fb258d29c19d,15720,palmeira d'oeste,SP
24999,56b18e2166679b8a959d72dd06da27f9,73c2643a0a458b49f58cea58833b192e,92120,canoas,RS


In [23]:
customers.shape

(25000, 5)

In [24]:
customers.info()

<class 'pandas.DataFrame'>
RangeIndex: 25000 entries, 0 to 24999
Data columns (total 5 columns):
 #   Column                    Non-Null Count  Dtype
---  ------                    --------------  -----
 0   customer_id               25000 non-null  str  
 1   customer_unique_id        25000 non-null  str  
 2   customer_zip_code_prefix  25000 non-null  int64
 3   customer_city             25000 non-null  str  
 4   customer_state            25000 non-null  str  
dtypes: int64(1), str(4)
memory usage: 2.8 MB


In [25]:
customers.isna().sum()

customer_id                 0
customer_unique_id          0
customer_zip_code_prefix    0
customer_city               0
customer_state              0
dtype: int64

In [26]:
customers["customer_id"].isna().sum()

np.int64(0)

In [27]:
customers["customer_state"].nunique()

27

In [28]:
customers.duplicated().sum()

np.int64(0)

In [29]:
customers["customer_id"].duplicated().sum()

np.int64(0)

### 3) Profiling table [items]
### note
- One row represents one item line within an order. 
- The Items table has 28,343 rows and 7 columns. 
- It has no missing values or fully duplicated rows. 
- The range of shipping_limit_date is from 2016 to 2020, which seems reasonable. However, the ranges of price (2.20 to 6,735.00) and freight_value (0 to 339.59) may need further inspection for possible unusual values.

In [30]:
items.head()

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87
1,00048cc3ae777c65dbb7d2a0634bc1ea,1,ef92defde845ab8450f9d70c526ef70f,6426d21aca402a131fc0a5d0960a3c90,2017-05-23 03:55:27,21.90,12.69
2,00054e8431b9d7675808bcb819fb4a32,1,8d4f2bb7e93e6710a28f34fa83ee7d28,7040e82f899a04d1b434b795a43b4617,2017-12-14 12:10:31,19.90,11.85
3,0005a1a1728c9d785b8e2b08b904576c,1,310ae3c140ff94b03219ad0adc3c778f,a416b6a846a11724393025641d4edd5e,2018-03-26 18:31:29,145.95,11.65
4,0009792311464db532ff765bf7b182ae,1,8cab8abac59158715e0d70a36c807415,530ec6109d11eaaf87999465c6afee01,2018-08-17 12:15:10,99.90,27.65


In [31]:
items.tail()

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
28338,fff7c4452f050315db1b3f24d9df5fcd,1,dd469c03ad67e201bc2179ef077dcd48,7e93a43ef30c4f03f38b393420bc753a,2017-06-07 17:05:23,736.00,20.91
28339,fff8287bbae429a99bb7e8c21d151c41,1,bee2e070c39f3dd2f6883a17a5f0da45,4e922959ae960d389249c378d1c939f5,2018-03-27 12:29:22,180.00,48.14
28340,fff8287bbae429a99bb7e8c21d151c41,2,bee2e070c39f3dd2f6883a17a5f0da45,4e922959ae960d389249c378d1c939f5,2018-03-27 12:29:22,180.00,48.14
28341,fffbee3b5462987e66fb49b1c5411df2,1,6f0169f259bb0ff432bfff7d829b9946,213b25e6f54661939f11710a6fddb871,2018-06-28 09:58:03,119.85,20.03
28342,fffcd46ef2263f404302a634eb57f7eb,1,32e07fd915822b0765e448c4dd74c828,f3c38ab652836d21de61fb8314b69182,2018-07-20 04:31:48,350.00,36.53


In [32]:
items.shape

(28343, 7)

In [33]:
items.info()

<class 'pandas.DataFrame'>
RangeIndex: 28343 entries, 0 to 28342
Data columns (total 7 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   order_id             28343 non-null  str           
 1   order_item_id        28343 non-null  int64         
 2   product_id           28343 non-null  str           
 3   seller_id            28343 non-null  str           
 4   shipping_limit_date  28343 non-null  datetime64[us]
 5   price                28343 non-null  float64       
 6   freight_value        28343 non-null  float64       
dtypes: datetime64[us](1), float64(2), int64(1), str(3)
memory usage: 4.1 MB


In [34]:
items.isna().sum()

order_id               0
order_item_id          0
product_id             0
seller_id              0
shipping_limit_date    0
price                  0
freight_value          0
dtype: int64

In [35]:
items.duplicated().sum()

np.int64(0)

In [36]:
items["shipping_limit_date"].min()

Timestamp('2016-10-08 10:34:01')

In [37]:
items["shipping_limit_date"].max()

Timestamp('2020-04-09 22:35:08')

In [38]:
items["price"].min()

np.float64(2.2)

In [39]:
items["price"].max()

np.float64(6735.0)

In [40]:
items["freight_value"].min()

np.float64(0.0)

In [41]:
items["freight_value"].max()

np.float64(339.59)

### 4) Profiling table [payments]
### note
- One row represents one payment transaction or payment method sequence within an order. 
- The Payments table has 26,154 rows and 5 columns. 
- There are no obvious data type or missing value issues. payment_type has 4 categories. 
- The payment_value ranges from 0 to 6,929.31, so the minimum and maximum values may need further inspection for possible unusual values.

In [42]:
payments.head()

,order_id,payment_sequential,payment_type,payment_installments,payment_value
0,25e8ea4e93396b6fa0d3dd708e76c1bd,1,credit_card,1,65.71
1,0573b5e23cbd798006520e1d5b4c6714,1,boleto,1,51.95
2,cf95215a722f3ebf29e6bbab87a29e61,1,credit_card,5,102.66
3,61059985a6fc0ad64e95d9944caacdad,1,credit_card,1,132.04
4,b2349a3f20dfbeef62e7b31baa22f84b,1,credit_card,3,136.71


In [43]:
payments.tail()

,order_id,payment_sequential,payment_type,payment_installments,payment_value
26149,0872d0faafe0cb56e6f2c594cca1522c,1,credit_card,2,127.92
26150,2e3d96522c8c378d4135c12a65c977aa,1,credit_card,4,47.62
26151,7159096c5aa9be77f7f0c26c01ee9793,1,credit_card,4,280.65
26152,b8b61059626efa996a60be9bb9320e10,1,credit_card,5,369.54
26153,28bbae6599b09d39ca406b747b6632b1,1,boleto,1,191.58


In [44]:
payments.shape

(26154, 5)

In [45]:
payments.info()

<class 'pandas.DataFrame'>
RangeIndex: 26154 entries, 0 to 26153
Data columns (total 5 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   order_id              26154 non-null  str    
 1   payment_sequential    26154 non-null  int64  
 2   payment_type          26154 non-null  str    
 3   payment_installments  26154 non-null  int64  
 4   payment_value         26154 non-null  float64
dtypes: float64(1), int64(2), str(2)
memory usage: 2.0 MB


In [46]:
payments.isna().sum()

order_id                0
payment_sequential      0
payment_type            0
payment_installments    0
payment_value           0
dtype: int64

In [47]:
payments.duplicated().sum()

np.int64(0)

In [48]:
payments["payment_type"].nunique()

4

In [49]:
payments["payment_type"].value_counts()

payment_type
credit_card    19302
boleto          4973
voucher         1488
debit_card       391
Name: count, dtype: int64

In [50]:
payments["payment_installments"].value_counts()

payment_installments
1     13147
2      3093
3      2665
4      1843
10     1372
5      1300
8      1095
6       955
7       437
9       170
12       31
15       14
24        7
18        6
13        5
20        5
21        2
11        2
14        2
16        2
17        1
Name: count, dtype: int64

In [51]:
payments["payment_value"].min()

np.float64(0.0)

In [52]:
payments["payment_value"].max()

np.float64(6929.31)

### 5) Profiling table [reviews]
### note
- One row represents one review record associated with an order. 
- The Reviews table has 24,934 rows and 7 columns. 
- The data types and missing values seem reasonable, and there are no fully duplicated rows. 
- review_id has 56 duplicated values, but review_id is not guaranteed to be unique in this table, so further inspection is needed before treating them as duplicate issues. 
- The ranges of review_creation_date and review_answer_timestamp also seem reasonable.

In [53]:
reviews.head()

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
0,228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,NaN,NaN,2018-02-17,2018-02-18 14:36:24
1,f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,NaN,Parabéns lojas lannister adorei comprar pela I...,2018-03-01,2018-03-02 10:26:53
2,7c6400515c67679fbee952a7525281ef,c31a859e34e3adac22f376954e19b39d,5,NaN,NaN,2018-08-14,2018-08-14 21:36:06
3,8670d52e15e00043ae7de4c01cc2fe06,b9bf720beb4ab3728760088589c62129,4,recomendo,aparelho eficiente. no site a marca do aparelh...,2018-05-22,2018-05-23 16:45:47
4,8e15a274d95600fa14f8be64e37a0e67,ff1581e08b3011021e7c7de592ddc81e,5,NaN,NaN,2018-03-24,2018-03-26 15:58:32


In [54]:
reviews.tail()

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
24929,6c5fa14b1cb0ede918c75b1fabb222a5,10a3f7e9bf88b7dc86a767fd278e5bd2,5,NaN,NaN,2017-11-21,2017-11-23 08:28:00
24930,586e77c5f02b95285a939083359f8241,5f4a120717a4d152577d7d2ec66c45be,5,NaN,NaN,2018-08-05,2018-08-06 09:48:41
24931,c2642a756d89f7e781ca0180829770bb,652050879a7da861024a1114f926958d,5,NaN,NaN,2018-06-08,2018-06-08 22:22:23
24932,3ea12859f456855149b564d1a221f87a,f62f01ec64c6b221f926ba560907d8f7,3,NaN,NaN,2017-04-05,2017-04-05 09:42:58
24933,af2dc0519de6e0720ef0c74292fb4114,d699c734a0b1c8111f2272a3f36d398c,5,NaN,NaN,2018-04-27,2018-04-30 01:18:57


In [55]:
reviews.shape

(24934, 7)

In [56]:
reviews.info()

<class 'pandas.DataFrame'>
RangeIndex: 24934 entries, 0 to 24933
Data columns (total 7 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   review_id                24934 non-null  str           
 1   order_id                 24934 non-null  str           
 2   review_score             24934 non-null  int64         
 3   review_comment_title     2926 non-null   str           
 4   review_comment_message   10324 non-null  str           
 5   review_creation_date     24934 non-null  datetime64[us]
 6   review_answer_timestamp  24934 non-null  datetime64[us]
dtypes: datetime64[us](2), int64(1), str(4)
memory usage: 3.6 MB


In [57]:
reviews.isna().sum()

review_id                      0
order_id                       0
review_score                   0
review_comment_title       22008
review_comment_message     14610
review_creation_date           0
review_answer_timestamp        0
dtype: int64

In [58]:
reviews.duplicated().sum()

np.int64(0)

In [59]:
reviews["review_id"].duplicated().sum()

np.int64(56)

In [60]:
reviews["review_creation_date"].min()

Timestamp('2016-10-15 00:00:00')

In [61]:
reviews["review_creation_date"].max()

Timestamp('2018-08-31 00:00:00')

In [62]:
reviews["review_answer_timestamp"].min()

Timestamp('2016-10-16 03:20:17')

In [63]:
reviews["review_answer_timestamp"].max()

Timestamp('2018-10-29 12:27:35')

### 6) Profiling table [products]
### note
- One row represents one product. 
- The Products table has 13,344 rows and 9 columns. 
- The data types and missing values seem reasonable, although product_photos_qty may be converted to int64 after handling its missing values. 
- product_category_name has 72 categories. 
- The ranges of the numerical columns generally seem reasonable, but a few maximum values for product dimensions and weight may need further investigation for possible unusual values.

In [64]:
products.head()

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,41d3672d4792049fa1779bb35283ed13,instrumentos_musicais,60.0,745.0,1.0,200.0,38.0,5.0,11.0
1,14aa47b7fe5c25522b47b4b29c98dcb9,cama_mesa_banho,54.0,630.0,1.0,1100.0,16.0,10.0,16.0
2,cf55509ea8edaaac1d28fdb16e48fc22,instrumentos_musicais,43.0,1827.0,3.0,250.0,17.0,7.0,17.0
3,6a2fb4dd53d2cdb88e0432f1284a004c,perfumaria,39.0,346.0,2.0,400.0,27.0,5.0,20.0
4,a0736b92e52f6cead290e30b578413b6,moveis_decoracao,56.0,296.0,2.0,1700.0,100.0,7.0,15.0


In [65]:
products.tail()

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
13339,dfec64aac9b864b2807a7be33222b75f,cama_mesa_banho,58.0,252.0,4.0,850.0,38.0,7.0,28.0
13340,3becff10d1deb92b02f2a1ee62a04524,informatica_acessorios,54.0,1520.0,2.0,6150.0,30.0,30.0,20.0
13341,c4e71b64511b959455e2107fe7859020,utilidades_domesticas,59.0,1371.0,2.0,200.0,18.0,15.0,15.0
13342,a0b7d5a992ccda646f2d34e418fff5a0,moveis_decoracao,45.0,67.0,2.0,12300.0,40.0,40.0,40.0
13343,83808703fc0706a22e264b9d75f04a2e,informatica_acessorios,60.0,156.0,2.0,700.0,31.0,13.0,20.0


In [66]:
products.shape

(13344, 9)

In [67]:
products.info()

<class 'pandas.DataFrame'>
RangeIndex: 13344 entries, 0 to 13343
Data columns (total 9 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   product_id                  13344 non-null  str    
 1   product_category_name       13114 non-null  str    
 2   product_name_lenght         13114 non-null  float64
 3   product_description_lenght  13114 non-null  float64
 4   product_photos_qty          13114 non-null  float64
 5   product_weight_g            13343 non-null  float64
 6   product_length_cm           13343 non-null  float64
 7   product_height_cm           13343 non-null  float64
 8   product_width_cm            13343 non-null  float64
dtypes: float64(7), str(2)
memory usage: 1.5 MB


In [68]:
products.isna().sum()

product_id                      0
product_category_name         230
product_name_lenght           230
product_description_lenght    230
product_photos_qty            230
product_weight_g                1
product_length_cm               1
product_height_cm               1
product_width_cm                1
dtype: int64

In [69]:
products.duplicated().sum()

np.int64(0)

In [70]:
products["product_id"].duplicated().sum()

np.int64(0)

In [71]:
products["product_category_name"].nunique()

72

In [72]:
products[["product_name_lenght",
"product_description_lenght",
"product_photos_qty", 
"product_weight_g",            
"product_length_cm",           
"product_height_cm",           
"product_width_cm"]].agg(["min","max"])

,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
min,5.0,4.0,1.0,2.0,7.0,2.0,7.0
max,76.0,3985.0,18.0,30000.0,105.0,105.0,118.0


### 7) Profiling table [sellers]
### note
- One row represents one seller. 
- The Sellers table has 2,234 rows and 4 columns. 
- There are no missing values. seller_zip_code_prefix is stored as an integer, but it represents a geographic code, so it should be converted to a string during cleaning. 
- seller_state has 21 unique categories.

In [73]:
sellers.head()

,seller_id,seller_zip_code_prefix,seller_city,seller_state
0,3442f8959a84dea7ee197c632cb2df15,13023,campinas,SP
1,d1b65fc7debc3361ea86b5f14c68d2e2,13844,mogi guacu,SP
2,c0f3eea2e14555b6faeea3dd58c1b1c3,4195,sao paulo,SP
3,e49c26c3edfa46d227d5121a6b6e4d37,55325,brejao,PE
4,1b938a7ec6ac5061a66a3766e0e75f90,16304,penapolis,SP


In [74]:
sellers.tail()

,seller_id,seller_zip_code_prefix,seller_city,seller_state
2229,725af026f869236a8dd0ceb98564d156,35530,claudio,MG
2230,7bac63f6603d382cc8d0832eb6c100a8,5713,sao paulo,SP
2231,f1fdf2d13186575751aa25876536d85c,5314,sao paulo,SP
2232,f8201cab383e484733266d1906e2fdfa,88137,palhoca,SC
2233,e603cf3fec55f8697c9059638d6c8eb5,96080,pelotas,RS


In [75]:
sellers.shape

(2234, 4)

In [76]:
sellers.info()

<class 'pandas.DataFrame'>
RangeIndex: 2234 entries, 0 to 2233
Data columns (total 4 columns):
 #   Column                  Non-Null Count  Dtype
---  ------                  --------------  -----
 0   seller_id               2234 non-null   str  
 1   seller_zip_code_prefix  2234 non-null   int64
 2   seller_city             2234 non-null   str  
 3   seller_state            2234 non-null   str  
dtypes: int64(1), str(3)
memory usage: 166.1 KB


In [77]:
sellers.isna().sum()

seller_id                 0
seller_zip_code_prefix    0
seller_city               0
seller_state              0
dtype: int64

In [78]:
sellers.duplicated().sum()

np.int64(0)

In [79]:
sellers["seller_id"].duplicated().sum()

np.int64(0)

In [80]:
sellers["seller_state"].nunique()

21

In [81]:
sellers["seller_state"].value_counts()

seller_state
SP    1362
PR     247
MG     170
SC     136
RJ     119
RS      84
DF      27
GO      26
ES      16
BA      13
CE       7
PE       6
RN       4
PB       4
MS       4
MT       3
RO       2
PI       1
SE       1
MA       1
PA       1
Name: count, dtype: int64

### 8) Profiling table [geo(geolocation_zip_prefix)]
### note
- One row represents one zip-code-prefix geographic summary. 
- The Geolocation table has 10,009 rows and 6 columns. 
- geolocation_zip_code_prefix is stored as an integer, but it represents a geographic code, so it should be converted to a string during cleaning. The other data types and missing values show no obvious issues.
- geolocation_state has 27 categories. 
- The ranges of latitude (geolocation_lat) and longitude (geolocation_lng) were also checked and may need further inspection to confirm that the geographic coordinates are reasonable.

In [82]:
geo.head()

,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state,geolocation_records
0,1001,-23.550190,-46.634024,sao paulo,SP,26
1,1004,-23.549799,-46.634757,sao paulo,SP,22
2,1005,-23.549456,-46.636733,sao paulo,SP,25
3,1006,-23.550102,-46.636137,sao paulo,SP,9
4,1008,-23.546002,-46.635886,sao paulo,SP,16


In [83]:
geo.tail()

,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state,geolocation_records
10004,99900,-27.886630,-52.228478,getulio vargas,RS,102
10005,99930,-27.914102,-52.258782,estacao,RS,32
10006,99950,-28.068959,-52.012068,tapejara,RS,153
10007,99960,-27.953722,-52.025511,charrua,RS,5
10008,99970,-28.343766,-51.874689,ciriaco,RS,21


In [84]:
geo.shape

(10009, 6)

In [85]:
geo.info()

<class 'pandas.DataFrame'>
RangeIndex: 10009 entries, 0 to 10008
Data columns (total 6 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   geolocation_zip_code_prefix  10009 non-null  int64  
 1   geolocation_lat              10009 non-null  float64
 2   geolocation_lng              10009 non-null  float64
 3   geolocation_city             10009 non-null  str    
 4   geolocation_state            10009 non-null  str    
 5   geolocation_records          10009 non-null  int64  
dtypes: float64(2), int64(2), str(2)
memory usage: 588.4 KB


In [86]:
geo.isna().sum()

geolocation_zip_code_prefix    0
geolocation_lat                0
geolocation_lng                0
geolocation_city               0
geolocation_state              0
geolocation_records            0
dtype: int64

In [87]:
geo.duplicated().sum()

np.int64(0)

In [88]:
geo["geolocation_state"].nunique()

27

In [89]:
geo["geolocation_state"].value_counts()

geolocation_state
SP    4143
MG     928
RJ     916
PR     594
RS     577
BA     412
SC     353
GO     296
DF     233
PE     232
CE     207
ES     198
MS     109
PA     108
MT     106
MA      93
PB      89
RN      82
PI      71
AL      66
SE      51
TO      48
RO      32
AM      31
AC      12
AP      11
RR      11
Name: count, dtype: int64

In [90]:
geo[["geolocation_lat", "geolocation_lng"]].agg(["min","max"])

,geolocation_lat,geolocation_lng
min,-33.689948,-68.749175
max,7.271649,-34.799347


### 9) Profiling table [translations]
### note
- One row represents one Portuguese to English product category translation. 
- The Translations table has 71 rows and 2 columns. 
- The data types and missing values show no obvious issues, and there are no fully duplicated rows.

In [91]:
translations.head()

,product_category_name,product_category_name_english
0,beleza_saude,health_beauty
1,informatica_acessorios,computers_accessories
2,automotivo,auto
3,cama_mesa_banho,bed_bath_table
4,moveis_decoracao,furniture_decor


In [92]:
translations.shape

(71, 2)

In [93]:
translations.info()

<class 'pandas.DataFrame'>
RangeIndex: 71 entries, 0 to 70
Data columns (total 2 columns):
 #   Column                         Non-Null Count  Dtype
---  ------                         --------------  -----
 0   product_category_name          71 non-null     str  
 1   product_category_name_english  71 non-null     str  
dtypes: str(2)
memory usage: 3.5 KB


In [94]:
translations.isna().sum()

product_category_name            0
product_category_name_english    0
dtype: int64

In [95]:
translations.duplicated().sum()

np.int64(0)

In [96]:
translations["product_category_name"].duplicated().sum()

np.int64(0)

### 10) Initial Data Profiling Summary

The Olist dataset contains nine tables with different row meanings and levels of granularity. Initial profiling was conducted to examine table dimensions, data types, missing values, duplicates, categorical values, and numerical ranges.

### Orders
- 25,000 rows and 8 columns.
- One row represents one order.
- Data types generally appear appropriate.
- Some date columns contain missing values and require further investigation.
- `order_id` is unique with no missing values.
- No fully duplicated rows were identified.
- `order_status` contains 8 categories.
- The purchase date range appears reasonable.

### Customers
- 25,000 rows and 5 columns.
- Each row represents a customer record for one order.
- No missing values or fully duplicated rows were identified.
- `customer_id` is unique, while `customer_unique_id` can repeat across orders.
- `customer_zip_code_prefix` is stored as an integer but represents a geographic code and may need to be converted to a string.

### Order Items
- 28,343 rows and 7 columns.
- One row represents one item line within an order.
- No missing values or fully duplicated rows were identified.
- The shipping date range appears reasonable.
- Some maximum values in `price` and `freight_value` may require further inspection.

### Payments
- 26,154 rows and 5 columns.
- One row represents one payment transaction or payment method sequence within an order.
- No obvious data type or missing value issues were identified.
- `payment_type` contains 4 categories.
- `payment_value` ranges from 0 to 6,929.31, so extreme values may require further inspection.

### Reviews
- 24,934 rows and 7 columns.
- One row represents one review record associated with an order.
- Data types and missing values generally appear reasonable.
- No fully duplicated rows were identified.
- `review_id` has 56 duplicated values. Further investigation is needed because `review_id` is not guaranteed to be unique.
- Review creation and answer timestamp ranges appear reasonable.

### Products
- 13,344 rows and 9 columns.
- One row represents one product.
- Data types and missing values generally appear reasonable.
- `product_photos_qty` may be converted to an integer after handling its missing values.
- `product_category_name` contains 72 categories.
- Some maximum values for product dimensions and weight may require further inspection.

### Sellers
- 2,234 rows and 4 columns.
- One row represents one seller.
- No missing values were identified.
- `seller_zip_code_prefix` is stored as an integer but represents a geographic code and may need to be converted to a string.
- `seller_state` contains 21 categories.

### Geolocation
- 10,009 rows and 6 columns.
- One row represents one zip-code-prefix geographic summary.
- No obvious missing value issues were identified.
- `geolocation_zip_code_prefix` should be converted from integer to string.
- `geolocation_state` contains 27 categories.
- Latitude and longitude ranges may require further inspection.

### Product Category Translation
- 71 rows and 2 columns.
- One row represents one Portuguese-to-English product category translation.
- No obvious data type or missing value issues were identified.
- No fully duplicated rows were identified.

### 11) Specific Data Quality Issues

### Additional Profiling Check
Categorical values were checked using `value_counts()`, and no obvious inconsistencies were identified.

### Issue 1: ZIP code columns are stored as integers
- **Table / Column:** Customers - `customer_zip_code_prefix`, Sellers - `seller_zip_code_prefix`, Geolocation - `geolocation_zip_code_prefix`
- **Finding:** ZIP code prefixes are stored as `int64`.
- **Concern:** ZIP codes are geographic identifiers rather than numerical measurements.
- **Initial plan:** Convert these columns to string type and verify the values after conversion.

### Issue 2: Missing delivery-related dates
- **Table / Column:** Orders - `order_approved_at`, `order_delivered_carrier_date`, `order_delivered_customer_date`
- **Finding:** These date columns contain missing values.
- **Concern:** The missing values may be related to order status rather than simple data errors.
- **Initial plan:** Compare the missing values with `order_status` before deciding how they should be handled.

### Issue 3: Potential unusual values in item price and freight value
- **Table / Column:** Order Items - `price`, `freight_value`
- **Finding:** `price` ranges from 2.20 to about 6,735, and `freight_value` ranges from 0 to 339.59.
- **Concern:** Some extreme values may represent unusual observations.
- **Initial plan:** Inspect the distributions and records with extreme values before deciding whether cleaning is necessary.

### Issue 4: Potential unusual payment values
- **Table / Column:** Payments - `payment_value`
- **Finding:** `payment_value` ranges from 0 to 6,929.31.
- **Concern:** Zero payments and very high payment values may require further investigation.
- **Initial plan:** Inspect records with zero and unusually high payment values and compare them with related order information.

### Issue 5: Repeated review IDs
- **Table / Column:** Reviews - `review_id`
- **Finding:** `review_id` contains 56 duplicated values.
- **Concern:** `review_id` is not guaranteed to be unique by itself, so these records should not automatically be treated as duplicates.
- **Initial plan:** Inspect the repeated `review_id` records and their associated `order_id` values before deciding whether any duplicate handling is needed.

### 12) Initial Cleaning Plan

- Convert ZIP code prefix columns in Customers, Sellers, and Geolocation from integer to string type.
- Investigate missing delivery-related dates in Orders by comparing them with `order_status` before deciding how to handle them.
- Inspect unusual values in `price`, `freight_value`, and `payment_value` before deciding whether cleaning is necessary.
- Investigate repeated `review_id` values with their associated `order_id` values before performing any duplicate handling.
- Keep the raw data unchanged and document any cleaning decisions before creating cleaned versions of the data.

## 4. Analytical Design and Data Preparation Plan

### A. Late delivery outcome + usuable population definition
- actual delivery date > estimated delivery date => late delivery

### Note

Late delivery can only be evaluated for orders that were actually delivered and have enough information to compare the actual delivery date with the estimated delivery date. I first reviewed order_status to understand the different order outcomes in the dataset. I then examined missing values in order_delivered_customer_date by order status to determine whether missing actual delivery dates were related to orders that had not been delivered. I also checked order_estimated_delivery_date for missing values because this date is required as the reference point for determining whether an order arrived late. Based on these checks, the late-delivery analysis will focus on delivered orders with both order_delivered_customer_date and order_estimated_delivery_date available.

In [97]:
orders

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,3b697a20d9e427646d92567910af6d57,355077684019f7f60a031656bd7262b8,delivered,2016-10-03 09:44:50,2016-10-06 15:50:54,2016-10-23 14:02:13,2016-10-26 14:02:13,2016-10-27
1,cd3b8574c82b42fc8129f6d502690c3e,7812fcebfc5e8065d31e1bb5f0017dae,delivered,2016-10-03 22:31:31,2016-10-04 10:19:23,2016-10-08 10:34:01,2016-10-14 16:08:00,2016-11-23
2,1aecadf4362edaca7fa033e882076c8d,e81a9f176936e3124dfd90c483bf3289,canceled,2016-10-04 10:05:45,2016-10-04 10:26:40,NaT,NaT,2016-11-24
3,3f72d2b757e725cd48a4726f831c7789,e0de521fbb397bd7e83d079fe66357b5,delivered,2016-10-04 13:15:46,2016-10-04 13:47:04,2016-10-10 16:45:50,2016-10-17 11:25:59,2016-11-28
4,63638a6806d67773f3adba8534553fff,16e14c1e6e050fe6730c961ff638ca23,delivered,2016-10-04 13:22:56,2016-10-04 13:47:45,2016-11-17 15:53:01,2016-11-25 13:17:37,2016-11-28
...,...,...,...,...,...,...,...,...
24995,21a00b08cbeb5716bbb66105e3dbd850,002b5342c72978cf0aba6aae1f5d5293,canceled,2018-09-10 17:51:25,NaT,NaT,NaT,2018-09-20
24996,87ae60ef8b08ae0e5f903cacb53a6904,fea4d57ed3a45455f89c25ef3dae8ee8,canceled,2018-09-11 16:45:54,NaT,NaT,NaT,2018-09-26
24997,ea844c92cf978ea23321fa7fe5871761,6ddc45ad08e6a2c885ca2255fc52d6df,canceled,2018-09-13 09:56:12,NaT,NaT,NaT,2018-09-27
24998,392ed9afd714e3c74767d0c4d3e3f477,2823ffda607a2316375088e0d00005ec,canceled,2018-09-29 09:13:03,NaT,NaT,NaT,2018-10-15


- 25,000 total orders -> 24,222 delivered orders -> 24,222 usable orders for late delivery analysis

In [98]:
orders["order_status"].value_counts()

order_status
delivered      24222
shipped          309
unavailable      167
canceled         154
invoiced          75
processing        71
approved           1
created            1
Name: count, dtype: int64

- count order_status when order_delivered_customer_date is missing
- order_delivered_customer_date <-> order_status relationship check
- check whether order_estimated_delivery_date has any missing data

In [99]:
orders["order_delivered_customer_date"].isna().sum()

np.int64(775)

In [100]:
orders[orders["order_delivered_customer_date"].isna()]["order_status"].value_counts()

order_status
shipped        309
unavailable    167
canceled       151
invoiced        75
processing      71
approved         1
created          1
Name: count, dtype: int64

In [101]:
orders["order_estimated_delivery_date"].isna().sum()

np.int64(0)

So far, usable population condition

- order_status == "delivered"
- when order_delivered_customer_date is not missing
- when order_estimated_delivery_date is not missing

In [102]:
usable_orders = orders[
    (orders["order_status"]=="delivered")
    & (orders["order_delivered_customer_date"].notna())
    & (orders["order_estimated_delivery_date"].notna())
]

usable_orders

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,3b697a20d9e427646d92567910af6d57,355077684019f7f60a031656bd7262b8,delivered,2016-10-03 09:44:50,2016-10-06 15:50:54,2016-10-23 14:02:13,2016-10-26 14:02:13,2016-10-27
1,cd3b8574c82b42fc8129f6d502690c3e,7812fcebfc5e8065d31e1bb5f0017dae,delivered,2016-10-03 22:31:31,2016-10-04 10:19:23,2016-10-08 10:34:01,2016-10-14 16:08:00,2016-11-23
3,3f72d2b757e725cd48a4726f831c7789,e0de521fbb397bd7e83d079fe66357b5,delivered,2016-10-04 13:15:46,2016-10-04 13:47:04,2016-10-10 16:45:50,2016-10-17 11:25:59,2016-11-28
4,63638a6806d67773f3adba8534553fff,16e14c1e6e050fe6730c961ff638ca23,delivered,2016-10-04 13:22:56,2016-10-04 13:47:45,2016-11-17 15:53:01,2016-11-25 13:17:37,2016-11-28
5,7033745709b7cf1bac7d2533663592de,7f0ca17bb33b230b47459437cf0682c7,delivered,2016-10-04 14:13:22,2016-10-04 14:46:48,2016-10-08 14:46:49,2016-10-11 14:46:49,2016-11-30
...,...,...,...,...,...,...,...,...
24986,9344079d5fbb7dd79af908b7eb667db7,be169d57b327a59f2db49cffdc6caf13,delivered,2018-08-28 15:11:49,2018-08-29 03:15:39,2018-08-29 09:47:00,2018-08-30 17:28:28,2018-09-03
24987,2c1f8fd9aa5f50ec17f4c95e52223a3a,0b0e8ff0c3d490e69901bd6f0efa190a,delivered,2018-08-28 15:53:09,2018-08-28 16:05:19,2018-08-29 19:11:00,2018-08-30 15:11:38,2018-09-06
24988,6d6783e9e03bc56ba0d4f018d45babe3,eb9f43bc3cb1915ba52ea755eddad3d4,delivered,2018-08-28 19:27:43,2018-08-28 19:43:57,2018-08-29 19:02:00,2018-08-30 18:22:47,2018-09-05
24989,d4fae577806d683110e00e18a5e181be,49a6ae8a95c6a78d90945b983ab1ecfc,delivered,2018-08-28 19:32:05,2018-08-29 03:15:36,2018-08-29 13:00:00,2018-08-30 18:32:36,2018-09-12


### B. Olist Tables + Candidate Variables

- **Late-delivery risk:** Which order characteristics may be related to late delivery?
- **Geographic / fulfillment:** Which data may help interpret geographic delivery patterns?

In [103]:
orders.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,3b697a20d9e427646d92567910af6d57,355077684019f7f60a031656bd7262b8,delivered,2016-10-03 09:44:50,2016-10-06 15:50:54,2016-10-23 14:02:13,2016-10-26 14:02:13,2016-10-27
1,cd3b8574c82b42fc8129f6d502690c3e,7812fcebfc5e8065d31e1bb5f0017dae,delivered,2016-10-03 22:31:31,2016-10-04 10:19:23,2016-10-08 10:34:01,2016-10-14 16:08:00,2016-11-23
2,1aecadf4362edaca7fa033e882076c8d,e81a9f176936e3124dfd90c483bf3289,canceled,2016-10-04 10:05:45,2016-10-04 10:26:40,NaT,NaT,2016-11-24
3,3f72d2b757e725cd48a4726f831c7789,e0de521fbb397bd7e83d079fe66357b5,delivered,2016-10-04 13:15:46,2016-10-04 13:47:04,2016-10-10 16:45:50,2016-10-17 11:25:59,2016-11-28
4,63638a6806d67773f3adba8534553fff,16e14c1e6e050fe6730c961ff638ca23,delivered,2016-10-04 13:22:56,2016-10-04 13:47:45,2016-11-17 15:53:01,2016-11-25 13:17:37,2016-11-28


In [116]:
candidate_tables = {
    "orders": orders,
    "customers": customers,
    "items": items,
    "products": products,
    "sellers": sellers,
    "geolocation": geo
}

for name, df in candidate_tables.items():
    print(f"\n{name}")
    print(df.columns.tolist())


orders
['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']

customers
['customer_id', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state']

items
['order_id', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value']

products
['product_id', 'product_category_name', 'product_name_lenght', 'product_description_lenght', 'product_photos_qty', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm']

sellers
['seller_id', 'seller_zip_code_prefix', 'seller_city', 'seller_state']

geolocation
['geolocation_zip_code_prefix', 'geolocation_lat', 'geolocation_lng', 'geolocation_city', 'geolocation_state', 'geolocation_records']


- **Orders**
  - Purchase timing
  - Delivery-related dates
  - Late-delivery outcome

- **Customers**
  - Customer state
  - Customer ZIP-code prefix

- **Order Items**
  - Item count
  - Seller
  - Product
  - Price
  - Freight value

- **Products**
  - Product category
  - Product weight
  - Product dimensions

- **Sellers**
  - Seller state
  - Seller ZIP-code prefix

- **Geolocation**
  - ZIP-code prefix
  - Latitude and longitude

- These variables provide potential information for both late-delivery risk and geographic delivery analysis.

### C. Variable Timing + Potential Leakage

- Candidate variables should be checked based on when the information becomes available.
- Variables used to identify delivery risk should not include information that becomes available only after the delivery outcome is known.
- Check the timing-related fields in the `orders` table.

In [117]:
orders[
    [
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date"
    ]
].head()

,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,2016-10-03 09:44:50,2016-10-06 15:50:54,2016-10-23 14:02:13,2016-10-26 14:02:13,2016-10-27
1,2016-10-03 22:31:31,2016-10-04 10:19:23,2016-10-08 10:34:01,2016-10-14 16:08:00,2016-11-23
2,2016-10-04 10:05:45,2016-10-04 10:26:40,NaT,NaT,2016-11-24
3,2016-10-04 13:15:46,2016-10-04 13:47:04,2016-10-10 16:45:50,2016-10-17 11:25:59,2016-11-28
4,2016-10-04 13:22:56,2016-10-04 13:47:45,2016-11-17 15:53:01,2016-11-25 13:17:37,2016-11-28


- **Potential pre-delivery information**
  - Purchase timing
  - Customer location
  - Seller location
  - Product characteristics
  - Item price and freight information
  - Estimated delivery date

- **Post-purchase / outcome information**
  - Actual customer delivery date
  - Final order status
  - Review score and review information

- `order_delivered_customer_date` is needed to define the late-delivery outcome, but it should not be used as a predictor of that outcome.
- Review information should also not be used to identify delivery risk because it is generated after the customer receives the order.

### D. Unit of Analysis + Aggregation Plan

- **Final unit of analysis:** One order
- `orders` will be the base table because one row represents one order.
- Supporting tables may contain multiple rows for the same `order_id`.
- Check the number of rows and unique orders before planning future integration.

In [118]:
print("Orders")
print("Rows:", len(orders))
print("Unique order IDs:", orders["order_id"].nunique())

print("\nItems")
print("Rows:", len(items))
print("Unique order IDs:", items["order_id"].nunique())

print("\nPayments")
print("Rows:", len(payments))
print("Unique order IDs:", payments["order_id"].nunique())

print("\nReviews")
print("Rows:", len(reviews))
print("Unique order IDs:", reviews["order_id"].nunique())

Orders
Rows: 25000
Unique order IDs: 25000

Items
Rows: 28343
Unique order IDs: 24791

Payments
Rows: 26154
Unique order IDs: 25000

Reviews
Rows: 24934
Unique order IDs: 24797


- `orders` has one row per order, while the supporting tables can contain multiple records for the same order.
- Therefore, supporting tables will need to be summarized to the order level before integration.

### E. Geographic + Fulfillment Feasibility

- Customer and seller locations may help identify geographic delivery patterns.
- Check the geographic coverage available in the dataset.

In [119]:
print("Customer states:", customers["customer_state"].nunique())
print("Seller states:", sellers["seller_state"].nunique())
print("Geolocation states:", geo["geolocation_state"].nunique())

Customer states: 27
Seller states: 21
Geolocation states: 27


In [120]:
customers["customer_state"].value_counts()

customer_state
SP    10478
RJ     3313
MG     2933
RS     1403
PR     1262
SC      860
BA      851
ES      524
DF      505
GO      498
PE      417
CE      329
PA      221
MT      212
MS      182
MA      160
PB      147
PI      130
RN      120
AL      107
SE      100
TO       74
RO       72
AM       42
AC       25
AP       20
RR       15
Name: count, dtype: int64

In [121]:
sellers["seller_state"].value_counts()

seller_state
SP    1362
PR     247
MG     170
SC     136
RJ     119
RS      84
DF      27
GO      26
ES      16
BA      13
CE       7
PE       6
RN       4
PB       4
MS       4
MT       3
RO       2
PI       1
SE       1
MA       1
PA       1
Name: count, dtype: int64

In [122]:
geo[
    [
        "geolocation_zip_code_prefix",
        "geolocation_lat",
        "geolocation_lng"
    ]
].head()

,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng
0,1001,-23.550190,-46.634024
1,1004,-23.549799,-46.634757
2,1005,-23.549456,-46.636733
3,1006,-23.550102,-46.636137
4,1008,-23.546002,-46.635886


- Customer and seller location data provide enough geographic information to examine delivery patterns across areas.
- These patterns can help identify areas with delivery challenges, but cannot determine an optimal facility location by themselves.

### F. Expected Relationships + Analytical Plan

- Examine whether order characteristics and geographic patterns are associated with late delivery.
- Use order-level information from the supporting tables after appropriate aggregation.
- Compare late-delivery patterns across customer and seller locations.
- Start with descriptive analysis before moving to later modeling.

## 5. Early Privacy, Ethics, and Limitations

### Privacy and Ethics

- Customer and seller geographic information should be used only for aggregated analysis, not to identify individuals.
- Geographic differences in late delivery should not automatically be interpreted as poor performance by customers, sellers, or specific areas.
- The results should be used to support further operational analysis rather than make decisions automatically.

### Limitations

- The dataset does not include factors such as facility cost, labor availability, warehouse capacity, or detailed transportation information.
- Missing data and differences in table structure may affect the final analytical dataset.
- Relationships found in the analysis should be interpreted as associations, not causation.
- Geographic delivery patterns alone cannot determine the best location for additional fulfillment capacity.

## Milestone 2 Starter: Integration, Transformation, and Preliminary ABT

Build a preliminary ABT. State exactly what one row means, what key defines one row, and which tables were joined or aggregated.

Critical integration reminders:

- Enrich item records with product, category, and seller attributes while they are still item-level.
- Aggregate item, payment, and review records separately to `order_id`.
- Join each order-level summary to `orders` with merge validation.
- Treat the order-level ABT as trustworthy only after its row count and unique key have been checked.

The starter below demonstrates this sequence. Revise the variables to fit your question, and explain why each retained variable is useful.

### Milestone 2 written workspace

Use this cell to document the ABT design required by the Canvas Milestone 2 page.

- Final or preliminary unit of analysis:
- Primary key for one row:
- Tables aggregated before joining:
- Tables joined directly:
- Row-count validation after each major merge:
- New variables created:
- Remaining data quality concerns:



In [105]:
# Milestone 2 code workspace
# Build or revise your ABT here.
# Add row-count checks and duplicate-key checks after each major merge.



In [106]:
products_enriched = products.merge(
    translations,
    on="product_category_name",
    how="left",
    validate="many_to_one",
)

item_detail = (
    items
    .merge(
        products_enriched[[
            "product_id", "product_category_name_english", "product_weight_g"
        ]],
        on="product_id",
        how="left",
        validate="many_to_one",
    )
    .merge(
        sellers[["seller_id", "seller_state"]],
        on="seller_id",
        how="left",
        validate="many_to_one",
    )
)

item_summary = (
    item_detail.groupby("order_id")
    .agg(
        order_item_count=("order_item_id", "count"),
        product_count=("product_id", "nunique"),
        seller_count=("seller_id", "nunique"),
        category_count=("product_category_name_english", "nunique"),
        seller_state_count=("seller_state", "nunique"),
        item_price_total=("price", "sum"),
        freight_total=("freight_value", "sum"),
        product_weight_mean_g=("product_weight_g", "mean"),
    )
    .reset_index()
)

payment_summary = (
    payments.groupby("order_id")
    .agg(
        payment_record_count=("payment_sequential", "count"),
        payment_total=("payment_value", "sum"),
        payment_installments_max=("payment_installments", "max"),
        payment_type_count=("payment_type", "nunique"),
    )
    .reset_index()
)

reviews_for_summary = reviews.assign(
    review_comment_present=reviews["review_comment_message"].notna().astype(int)
)
review_summary = (
    reviews_for_summary.groupby("order_id")
    .agg(
        review_record_count=("review_id", "size"),
        review_score_mean=("review_score", "mean"),
        review_comment_present=("review_comment_present", "max"),
    )
    .reset_index()
)

customer_geo = customers.merge(
    geo[[
        "geolocation_zip_code_prefix", "geolocation_lat", "geolocation_lng",
        "geolocation_records"
    ]],
    left_on="customer_zip_code_prefix",
    right_on="geolocation_zip_code_prefix",
    how="left",
    validate="many_to_one",
).rename(columns={
    "geolocation_lat": "customer_lat",
    "geolocation_lng": "customer_lng",
    "geolocation_records": "customer_geo_source_records",
})

abt = (
    orders
    .merge(customer_geo, on="customer_id", how="left", validate="one_to_one")
    .merge(item_summary, on="order_id", how="left", validate="one_to_one")
    .merge(payment_summary, on="order_id", how="left", validate="one_to_one")
    .merge(review_summary, on="order_id", how="left", validate="one_to_one")
)

abt["purchase_month"] = abt["order_purchase_timestamp"].dt.month
abt["purchase_dayofweek"] = abt["order_purchase_timestamp"].dt.dayofweek
abt["promised_delivery_days"] = (
    abt["order_estimated_delivery_date"] - abt["order_purchase_timestamp"]
).dt.days
abt["delivery_days"] = (
    abt["order_delivered_customer_date"] - abt["order_purchase_timestamp"]
).dt.days

delivery_outcome_observed = (
    abt["order_status"].eq("delivered")
    & abt["order_delivered_customer_date"].notna()
    & abt["order_estimated_delivery_date"].notna()
)
abt["delivered_late"] = pd.Series(pd.NA, index=abt.index, dtype="Int64")
abt.loc[delivery_outcome_observed, "delivered_late"] = (
    abt.loc[delivery_outcome_observed, "order_delivered_customer_date"]
    > abt.loc[delivery_outcome_observed, "order_estimated_delivery_date"]
).astype(int)

abt["high_review_score"] = pd.Series(pd.NA, index=abt.index, dtype="Int64")
review_outcome_observed = abt["review_score_mean"].notna()
abt.loc[review_outcome_observed, "high_review_score"] = (
    abt.loc[review_outcome_observed, "review_score_mean"] >= 4
).astype(int)


In [107]:
abt_validation = pd.Series({
    "orders_before_merges": len(orders),
    "abt_rows_after_merges": len(abt),
    "unique_order_ids": abt["order_id"].nunique(),
    "duplicate_order_ids": abt["order_id"].duplicated().sum(),
    "orders_with_item_summary": abt["order_item_count"].notna().sum(),
    "orders_with_payment_summary": abt["payment_total"].notna().sum(),
    "orders_with_review_summary": abt["review_score_mean"].notna().sum(),
    "orders_with_observed_delivery_target": abt["delivered_late"].notna().sum(),
    "orders_with_observed_review_target": abt["high_review_score"].notna().sum(),
})

display(abt_validation.to_frame("value"))
display(abt.head())
display(abt.isna().mean().sort_values(ascending=False).head(15).to_frame("missing_rate"))


,value
orders_before_merges,25000
abt_rows_after_merges,25000
unique_order_ids,25000
duplicate_order_ids,0
orders_with_item_summary,24791
orders_with_payment_summary,25000
orders_with_review_summary,24797
orders_with_observed_delivery_target,24222
orders_with_observed_review_target,24797


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,geolocation_zip_code_prefix,customer_lat,customer_lng,customer_geo_source_records,order_item_count,product_count,seller_count,category_count,seller_state_count,item_price_total,freight_total,product_weight_mean_g,payment_record_count,payment_total,payment_installments_max,payment_type_count,review_record_count,review_score_mean,review_comment_present,purchase_month,purchase_dayofweek,promised_delivery_days,delivery_days,delivered_late,high_review_score
0,3b697a20d9e427646d92567910af6d57,355077684019f7f60a031656bd7262b8,delivered,2016-10-03 09:44:50,2016-10-06 15:50:54,2016-10-23 14:02:13,2016-10-26 14:02:13,2016-10-27,32ea3bdedab835c3aa6cb68ce66565ef,4106,sao paulo,SP,4106.0,-23.581451,-46.635029,45.0,1.0,1.0,1.0,1.0,1.0,29.90,15.56,300.0,1,45.46,1,1,1.0,4.0,1.0,10,0,23,23.0,0,1
1,cd3b8574c82b42fc8129f6d502690c3e,7812fcebfc5e8065d31e1bb5f0017dae,delivered,2016-10-03 22:31:31,2016-10-04 10:19:23,2016-10-08 10:34:01,2016-10-14 16:08:00,2016-11-23,87776adb449c551e74c13fc34f036105,12030,taubate,SP,12030.0,-23.032142,-45.570461,218.0,1.0,1.0,1.0,1.0,1.0,29.99,10.96,9000.0,1,40.95,4,1,1.0,5.0,0.0,10,0,50,10.0,0,1
2,1aecadf4362edaca7fa033e882076c8d,e81a9f176936e3124dfd90c483bf3289,canceled,2016-10-04 10:05:45,2016-10-04 10:26:40,NaT,NaT,2016-11-24,823c47d4abda1f8ce7568145f76c2b85,13175,sumare,SP,13175.0,-22.817992,-47.241101,173.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,50.74,3,1,1.0,2.0,0.0,10,1,50,NaN,<NA>,0
3,3f72d2b757e725cd48a4726f831c7789,e0de521fbb397bd7e83d079fe66357b5,delivered,2016-10-04 13:15:46,2016-10-04 13:47:04,2016-10-10 16:45:50,2016-10-17 11:25:59,2016-11-28,0829f7df6577d5a4b65439bea701405f,31210,belo horizonte,MG,31210.0,-19.906712,-43.948897,156.0,1.0,1.0,1.0,1.0,1.0,249.90,17.59,610.0,1,267.49,10,1,1.0,3.0,0.0,10,1,54,12.0,0,0
4,63638a6806d67773f3adba8534553fff,16e14c1e6e050fe6730c961ff638ca23,delivered,2016-10-04 13:22:56,2016-10-04 13:47:45,2016-11-17 15:53:01,2016-11-25 13:17:37,2016-11-28,df2988ba3ed226b10521a0e4da849b61,23092,rio de janeiro,RJ,23092.0,-22.870856,-43.543849,217.0,1.0,1.0,1.0,1.0,1.0,67.90,18.98,2100.0,7,86.88,1,1,1.0,5.0,1.0,10,1,54,51.0,0,1


,missing_rate
delivered_late,0.03112
delivery_days,0.03100
order_delivered_customer_date,0.03100
order_delivered_carrier_date,0.01808
product_weight_mean_g,0.00844
seller_count,0.00836
order_item_count,0.00836
freight_total,0.00836
item_price_total,0.00836
seller_state_count,0.00836


### SQL Validation Connected to the ABT

The project milestone asks for database or SQL work that supports the workflow. This in-memory SQLite check verifies the order-level ABT row count, key uniqueness, and target eligibility without creating another permanent database file.


In [108]:
import sqlite3

with sqlite3.connect(":memory:") as connection:
    abt.to_sql("olist_order_abt", connection, if_exists="replace", index=False)
    sql_abt_check = pd.read_sql_query(
        '''
        SELECT
            COUNT(*) AS abt_rows,
            COUNT(DISTINCT order_id) AS unique_order_ids,
            SUM(CASE WHEN delivered_late IS NOT NULL THEN 1 ELSE 0 END) AS delivery_target_rows,
            SUM(CASE WHEN high_review_score IS NOT NULL THEN 1 ELSE 0 END) AS review_target_rows
        FROM olist_order_abt
        ''',
        connection,
    )

display(sql_abt_check)


,abt_rows,unique_order_ids,delivery_target_rows,review_target_rows
0,25000,25000,24222,24797


### Data Dictionary Starter

Complete the text fields for every column you keep in the final ABT. Remove unused columns before finalizing the dictionary so the documentation and exported ABT stay aligned.


In [109]:
data_dictionary = pd.DataFrame({
    "column_name": abt.columns,
    "data_type": [str(abt[column].dtype) for column in abt.columns],
    "description": "",
    "source": "",
    "transformation_or_derivation": "",
    "notes_or_concerns": "",
})

# Hint: use .loc[] to document one column at a time, following the Week 11 pattern.
data_dictionary.head()


,column_name,data_type,description,source,transformation_or_derivation,notes_or_concerns
0,order_id,str,,,,
1,customer_id,str,,,,
2,order_status,str,,,,
3,order_purchase_timestamp,datetime64[us],,,,
4,order_approved_at,datetime64[us],,,,


## Milestone 3 Starter: Prepared Dataset and Initial Model Application

Choose a target/outcome and features. Start simple. Your first model should be correct and interpretable before it becomes complex.

Modeling reminders:

- If you model `delivered_late`, use only rows where the nullable target is defined; undelivered orders do not reveal a normal late/on-time outcome.
- For a prediction made at purchase time, do not use actual delivery duration, final order status, review fields, or other post-purchase outcomes as predictors.
- If you model `high_review_score`, use only rows with an observed review and do not use review-derived fields as predictors.
- Check class balance, keep a reproducible train/test split, and interpret precision and recall when the positive class is uncommon.
- Explain the intended decision and prediction timing before choosing features.


### Read the Canvas milestone page first

Before you start this section, read the matching Canvas milestone assignment page. The Canvas page is the authoritative checklist for what you need to submit, how the milestone is graded, and what evidence should appear in your notebook/report. Use this starter section as a workspace, not as a replacement for the Canvas instructions.



### Milestone 3 written workspace

Use this cell to document the prepared dataset and initial model/analysis required by the Canvas Milestone 3 page.

- Target/outcome:
- Feature variables:
- Rows included/excluded and why:
- Train/test split or evaluation plan:
- Baseline model or analysis:
- Initial metrics/results:
- Interpretation limits:



In [110]:
# Milestone 3 code workspace
# Suggested sequence:
# 1. State the prediction or analysis timing in markdown.
# 2. Filter to rows where your target is defined.
# 3. Select only features available at that time.
# 4. Check class balance or the target distribution.
# 5. Create a reproducible train/test split when modeling.
# 6. Fit a simple baseline and evaluate it on test data.


In [111]:
# Example placeholder. Replace with your chosen target and feature set.
# target = 'your_target_column'
# features = ['feature_1', 'feature_2']
# model_df = abt[[target] + features].dropna()

abt.describe(include='all').T.head(25)

,count,unique,top,freq,mean,min,25%,50%,75%,max,std
order_id,25000,25000,3b697a20d9e427646d92567910af6d57,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
customer_id,25000,25000,355077684019f7f60a031656bd7262b8,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
order_status,25000,8,delivered,24222,NaN,NaN,NaN,NaN,NaN,NaN,NaN
order_purchase_timestamp,25000,NaN,NaN,NaN,2017-12-31 22:18:29.561480,2016-10-03 09:44:50,2017-09-13 11:52:50,2018-01-19 19:08:12.500000,2018-05-05 11:17:57,2018-10-01 15:30:09,NaN
order_approved_at,24961,NaN,NaN,NaN,2018-01-01 06:46:13.413765,2016-10-04 10:19:23,2017-09-13 19:55:11,2018-01-20 09:08:59,2018-05-05 11:50:40,2018-09-03 17:40:06,NaN
order_delivered_carrier_date,24548,NaN,NaN,NaN,2018-01-05 06:50:25.146529,2016-10-08 10:34:01,2017-09-18 18:01:51.750000,2018-01-24 20:01:44.500000,2018-05-08 14:21:00,2018-09-04 15:25:00,NaN
order_delivered_customer_date,24225,NaN,NaN,NaN,2018-01-14 19:15:33.384024,2016-10-11 14:46:49,2017-09-26 19:04:55,2018-02-03 03:49:26,2018-05-16 12:26:50,2018-09-27 02:24:33,NaN
order_estimated_delivery_date,25000,NaN,NaN,NaN,2018-01-24 17:19:26.976000,2016-10-27 00:00:00,2017-10-04 00:00:00,2018-02-15 00:00:00,2018-05-28 00:00:00,2018-10-23 00:00:00,NaN
customer_unique_id,25000,24764,8d50f5eadf50201ccdcedfb9e2ac8455,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN
customer_zip_code_prefix,25000.0,NaN,NaN,NaN,34966.35068,1004.0,11431.75,24326.0,58037.0,99970.0,29706.603087


## Final Project Starter: Interpretation, Ethics, and Recommendations

Use this section to connect your analysis back to a business or research decision.

Address:

- Main finding
- What the model/analysis can and cannot support
- Limitations
- Ethics or responsible use concerns
- 2-3 cautious recommendations

### Read the Canvas final submission page first

Before completing this section, read the Canvas final submission assignment page. The Canvas page is the authoritative checklist for the final notebook, HTML report, required files, and submission format. Use this section to organize your final evidence and interpretation.



### Final report written workspace

Use this cell to draft the final interpretation required by the Canvas final submission page.

- Main finding:
- Evidence supporting the finding:
- Business or research implication:
- Limitations:
- Ethics/responsible use concerns:
- Recommendations:
- What you would improve with more time:



In [112]:
# Final project code workspace
# Save final datasets, figures, or tables here when they are ready.
# Recommended filenames:
# final_abt_path = OUTPUT_DIR / "olist_final_abt.csv"
# dictionary_path = OUTPUT_DIR / "olist_data_dictionary.csv"
# metrics_path = OUTPUT_DIR / "olist_model_metrics.csv"
# report_path = REPORTS_DIR / "olist_final_report.html"


In [113]:
# Save final deliverables only after your ABT and documentation are complete.
# abt.to_csv(OUTPUT_DIR / "olist_final_abt.csv", index=False)
# data_dictionary.to_csv(OUTPUT_DIR / "olist_data_dictionary.csv", index=False)

# Before final submission, restart the kernel, run all cells, and export the notebook to HTML.
